In [1]:
%%pyspark

# Get all the files under the ADLS folder and create a list of file paths
file_list = mssparkutils.fs.ls("abfss://enriched@storage_account.dfs.core.windows.net/")

# Read each file and create a DataFrame
for file_path in file_list:
    print(file_path.path)
    df = spark.read.format("delta").load(file_path.path)
    # Create a temporary table:
    df.createOrReplaceTempView(file_path.name)

StatementMeta(SparkPool01, 1, 2, Finished, Available, Finished, False)

abfss://enriched@storage_account.dfs.core.windows.net/salesCustomer
abfss://enriched@storage_account.dfs.core.windows.net/salesCustomerAddress
abfss://enriched@storage_account.dfs.core.windows.net/salesOrderDetail
abfss://enriched@storage_account.dfs.core.windows.net/salesOrderHeader
abfss://enriched@storage_account.dfs.core.windows.net/salesProduct
abfss://enriched@storage_account.dfs.core.windows.net/salesProductCategory


In [2]:
views = spark.sql("SHOW VIEWS")
display(views)

StatementMeta(SparkPool01, 1, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ec31dab-866d-4df1-aeb9-d77184762de8)

In [3]:
from pyspark.sql.functions import monotonically_increasing_id
#Create DimCustomer
df_dimCustomer = spark.sql("select sc.* , sca.AddressID, sca.AddressType from salesCustomer sc join salescustomeraddress sca on sc.customerid = sca.customerid")


#Add surrogate key as the first column
df_dimCustomer_with_surrogate_key = df_dimCustomer.withColumn("CustomerIDKey", monotonically_increasing_id())\
    .select(
        "CustomerIDKey",  # Select the surrogate key column first
        *[column for column in df_dimCustomer.columns if column != "CustomerIDKey"]  # Select the remaining columns in their original order
    )

df_dimCustomer_with_surrogate_key.createOrReplaceTempView('dimCustomer')

StatementMeta(SparkPool01, 1, 4, Finished, Available, Finished, False)

In [4]:
from pyspark.sql.functions import monotonically_increasing_id
#Create dimProduct
df_dimProduct = spark.sql("select sp.*, spc.ParentProductCategoryID, spc.Name as ProductCategoryName from salesproduct sp join salesproductcategory spc on sp.ProductCategoryID = spc.ProductCategoryID")

#Add surrogate key as the first column
df_dimProduct_with_surrogate_key = df_dimProduct.withColumn("ProductIDKey", monotonically_increasing_id())\
    .select(
        "ProductIDKey",  # Select the surrogate key column first
        *[column for column in df_dimProduct.columns if column != "ProductIDKey" and column!="spc.Name"]  # Select the remaining columns in their original order
    )

df_dimProduct_with_surrogate_key.createOrReplaceTempView('dimProduct')

StatementMeta(SparkPool01, 1, 5, Finished, Available, Finished, False)

In [6]:
from pyspark.sql.functions import expr
 
# Define the start and end dates for your DimDate table
start_date = "2000-01-01"
end_date = "2024-12-31"
 
# Create a DataFrame with a range of dates
df_dimDate = spark.range(0, (spark.sql("SELECT datediff('{0}', '{1}')".format(end_date, start_date)).collect()[0][0])+1) \
    .selectExpr("CAST(id AS INT) AS id") \
    .selectExpr("date_add('{0}', id) AS Date".format(start_date))
 
# Extract different date components
df_dimDate = df_dimDate \
    .withColumn("Year", expr("year(Date)")) \
    .withColumn("Month", expr("month(Date)")) \
    .withColumn("DayOfMonth", expr("dayofmonth(Date)")) \
    .withColumn("DayOfYear", expr("dayofyear(Date)")) \
    .withColumn("WeekOfYear", expr("weekofyear(Date)")) \
    .withColumn("DayOfWeek", expr("dayofweek(Date)")) \
    .withColumn("Quarter", expr("quarter(Date)"))

StatementMeta(SparkPool01, 1, 7, Finished, Available, Finished, False)

## Write dataframes to DataLake Gold Layer as Tables

In [7]:
# Write dimProduct table into DataLake Gold layer as Tables

path="abfss://curated@storage_account.dfs.core.windows.net/"

tableName="dimProduct"

df_dimProduct_with_surrogate_key.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName)


StatementMeta(SparkPool01, 1, 8, Finished, Available, Finished, False)

In [8]:
# Write dimCustomer table into DataLake Gold layer as Tables

path="abfss://curated@storage_account.dfs.core.windows.net/"

tableName="dimCustomer"

df_dimCustomer_with_surrogate_key.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName)


StatementMeta(SparkPool01, 1, 9, Finished, Available, Finished, False)

In [9]:
# Create factSales

df_factSales = spark.sql("select dp.ProductIDKey, ds.CustomerIDKey, soh.*, sod.OrderQty, sod.ProductID, sod.UnitPrice, sod.UnitPriceDiscount, sod.LineTotal from salesorderheader soh join salesorderdetail sod on soh.SalesOrderID = sod.SalesOrderID LEFT JOIN dimProduct dp ON sod.ProductID = dp.ProductID LEFT JOIN dimCustomer ds ON soh.CustomerID = ds.CustomerID")


StatementMeta(SparkPool01, 1, 10, Finished, Available, Finished, False)

In [10]:
# Write dimDate table into DataLake Gold layer as Tables

path="abfss://curated@storage_account.dfs.core.windows.net/"

tableName="dimDate"

df_dimDate.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName + "/")

StatementMeta(SparkPool01, 1, 11, Finished, Available, Finished, False)

In [11]:
# Write factSales table into DataLake Gold layer as Tables

path="abfss://curated@storage_account.dfs.core.windows.net/"

tableName="factSales"

df_factSales.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName)


StatementMeta(SparkPool01, 1, 12, Finished, Available, Finished, False)